In [1]:
import time
notebook_start = time.perf_counter()

%pip install -e /home/darshan/A6/PCSAFT_cDFT/thermoift

Obtaining file:///home/darshan/A6/PCSAFT_cDFT/thermoift
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for thermoift (pyproject.toml) ... done
  Created wheel for thermoift: filename=thermoift-0.2.0-0.editable-py3-none-any.whl size=1367 sha256=49ad06cc8dd338ef2637b65d2f2243be907be263a61c47dff94dcaefac90ff79
  Stored in directory: /tmp/pip-ephem-wheel-cache-18xtdyu_/wheels/fd/2f/c4/54a2ee5cd16a9bf5b183bbe5c28d1b3ba4926fb0261a13e1e4
Successfully built thermoift
  Attempting uninstall: thermoift
    Found existing installation: thermoift 0.2.0
    Uninstalling thermoift-0.2.0:
      Successfully uninstalled thermoift-0.2.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from thermoift import FeedsBuilder
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("CSV_feeds")
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
# ── Configuration ──
COMPONENTS  = ["CO2", "H2", "Ar", "N2", "CH4", "O2", "CO", "H2S"]
CO2_LEVELS  = (0.95, 0.96, 0.97, 0.98, 0.99)
RNG_TYPE        = "PCG64"
SEED            = 58
RANDOM_SEED     = SEED
INDUSTRIAL_SEED = SEED + 1
COMBINED_SEED   = SEED + 2

def make_builder(seed=SEED):
    return FeedsBuilder(rng_type=RNG_TYPE, seed=seed)

builder = make_builder()

### 1. Random feeds (Dirichlet sampling)

In [4]:
builder_random = make_builder(RANDOM_SEED)

df_random = builder_random.generate_random_feeds(
    components=COMPONENTS,
    mixture_sizes=(2, 3),
    co2_levels=CO2_LEVELS,
    n_random_samples=200,
)
print(f"Random feeds: {len(df_random)}")
df_random.head()

Random feeds: 28000


,CO2,H2,Ar,N2,CH4,O2,CO,H2S,mixture_size,active_components,feed_source,template_name
0,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
1,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
2,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
3,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet
4,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",random,dirichlet


### 2. Systematic feeds (grid at fixed step)

In [5]:
df_systematic = builder.generate_systematic_feeds(
    components=COMPONENTS,
    mixture_sizes=(2, 3),
    co2_levels=CO2_LEVELS,
    step=0.01,
)
print(f"Systematic feeds: {len(df_systematic)}")
df_systematic.head()

Systematic feeds: 455


,CO2,H2,Ar,N2,CH4,O2,CO,H2S,mixture_size,active_components,feed_source,template_name
0,0.95,0.05,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
1,0.96,0.04,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
2,0.97,0.03,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
3,0.98,0.02,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid
4,0.99,0.01,0.0,0.0,0.0,0.0,0.0,0.0,2,"CO2,H2",systematic,systematic_grid


### 3. Industrial feeds (bounds + ranges templates)

In [6]:
builder_industrial = make_builder(INDUSTRIAL_SEED)
builder_industrial.load_industrial_templates()

df_ind_bounds = builder_industrial.industrial_feeds_from_bounds(
    components=COMPONENTS,
    co2_levels=CO2_LEVELS,
)

df_ind_ranges = builder_industrial.industrial_feeds_from_ranges(
    components=COMPONENTS,
)

df_industrial = pd.concat([df_ind_bounds, df_ind_ranges], ignore_index=True)
print(f"Industrial feeds: {len(df_industrial)}  (bounds={len(df_ind_bounds)}, ranges={len(df_ind_ranges)})")
df_industrial.head()

Industrial feeds: 320  (bounds=180, ranges=140)


,CO2,H2,Ar,N2,CH4,O2,CO,H2S,feed_source,template_name,mixture_size,active_components
0,0.95,0.002273,0.007310,0.033833,0.006565,0.000001,0.000011,0.000007,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
1,0.95,0.022397,0.000164,0.001404,0.025985,0.000004,0.000019,0.000026,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
2,0.95,0.001642,0.019119,0.001904,0.027319,0.000002,0.000005,0.000009,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
3,0.95,0.020807,0.016937,0.010896,0.001232,0.000009,0.000026,0.000093,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
4,0.95,0.004155,0.018870,0.025527,0.001362,0.000009,0.000030,0.000047,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"


### 4. Combined feed (70% random / 20% systematic / 10% industrial)

In [7]:
builder_combined = make_builder(COMBINED_SEED)

df_all = builder_combined.combine_feeds_random_systematic_industrial(
    components=COMPONENTS,
    n_total_target=1000,
    co2_levels=CO2_LEVELS,
)

print(f"Total feeds: {len(df_all)}")
print()
print(df_all["feed_source"].value_counts())
print()
df_all.head(10)

Total feeds: 1000

feed_source
random              700
systematic          200
industrial_bound     55
industrial_range     45
Name: count, dtype: int64



,feed_source,template_name,mixture_size,active_components,CO2,H2,Ar,N2,CH4,O2,CO,H2S
0,random,dirichlet,5,"CO2,N2,CH4,O2,CO",0.99,0.000000,0.000000,0.002344,0.002533,0.003358,0.001765,0.000000
1,random,dirichlet,2,"CO2,O2",0.99,0.000000,0.000000,0.000000,0.000000,0.010000,0.000000,0.000000
2,random,dirichlet,6,"CO2,Ar,N2,CH4,CO,H2S",0.99,0.000000,0.002285,0.004233,0.000081,0.000000,0.002823,0.000577
3,random,dirichlet,4,"CO2,O2,CO,H2S",0.98,0.000000,0.000000,0.000000,0.000000,0.002101,0.001467,0.016431
4,random,dirichlet,6,"CO2,H2,Ar,CH4,O2,CO",0.96,0.019935,0.000013,0.000000,0.018088,0.001097,0.000867,0.000000
5,random,dirichlet,7,"CO2,H2,Ar,N2,O2,CO,H2S",0.97,0.004878,0.000690,0.001044,0.000000,0.003905,0.010473,0.009009
6,random,dirichlet,3,"CO2,N2,CH4",0.99,0.000000,0.000000,0.006555,0.003445,0.000000,0.000000,0.000000
7,random,dirichlet,5,"CO2,CH4,O2,CO,H2S",0.96,0.000000,0.000000,0.000000,0.009375,0.010131,0.013434,0.007060
8,random,dirichlet,2,"CO2,N2",0.97,0.000000,0.000000,0.030000,0.000000,0.000000,0.000000,0.000000
9,random,dirichlet,6,"CO2,H2,CH4,O2,CO,H2S",0.97,0.004338,0.000000,0.000000,0.004688,0.006216,0.003267,0.011491


### 5. Save all feeds

In [8]:
# Save individual feeds
df_random.to_csv(OUTPUT_DIR / "random_CO2_H2_Ar.csv", index=False)
df_systematic.to_csv(OUTPUT_DIR / "systematic_CO2_H2_Ar.csv", index=False)
df_industrial.to_csv(OUTPUT_DIR / "industrial_CO2_H2_Ar.csv", index=False)

# Save combined feed
df_all.to_csv(OUTPUT_DIR / "combined_CO2_H2_Ar.csv", index=False)

print(f"Saved to {OUTPUT_DIR}/:")
print(f"  random_CO2_H2_Ar.csv       : {len(df_random)} rows")
print(f"  systematic_CO2_H2_Ar.csv   : {len(df_systematic)} rows")
print(f"  industrial_CO2_H2_Ar.csv   : {len(df_industrial)} rows")
print(f"  combined_CO2_H2_Ar.csv     : {len(df_all)} rows")

elapsed = time.perf_counter() - notebook_start
print(f"\nNotebook runtime: {elapsed:.1f}s")

Saved to CSV_feeds/:
  random_CO2_H2_Ar.csv       : 28000 rows
  systematic_CO2_H2_Ar.csv   : 455 rows
  industrial_CO2_H2_Ar.csv   : 320 rows
  combined_CO2_H2_Ar.csv     : 1000 rows

Notebook runtime: 4.1s
